# Package

In [ ]:
%pip install pandas numpy matplotlib seaborn

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import pandas as pd
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Ajoute la racine du projet (dossier parent du notebook) au chemin de recherche de Python
PROJECT_DIR = Path.cwd().parent
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from Scripts.utils_segment import afficher_tableau, resume_variables, tableau_cible, graphique_taux_defaut

# Configuration de l'affichage Pandas pour tout voir clairement
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: "%.3f" % x)

# Données

In [ ]:

# Définition du chemin racine du projet (dossier parent de 'Notebooks')
PROJECT_DIR = Path.cwd().parent.parent if Path.cwd().name == "Notebooks" else Path.cwd()

# Définition des dossiers clés
DATA_DIR = PROJECT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "raw"

# Chemin vers le fichier SAS
sas_file_path = RAW_DATA_DIR / "_01_99__ensai_par_final.sas7bdat"

In [ ]:
# Chargement du fichier SAS7BDAT dans un DataFrame
df = pd.read_sas(sas_file_path, encoding="utf-8")

In [ ]:
# VUE D'ENSEMBLE ET DIMENSIONS ---
print("=" * 60)
print(f"DIMENSIONS DU DATASET : {df.shape[0]} lignes | {df.shape[1]} colonnes")
print("=" * 60)

# Vérification des doublons sur l'ensemble de la base
nb_doublons = df.duplicated().sum()
print(f"Nombre de lignes totalement doublonnées : {nb_doublons}")
print("-" * 60)

# AUDIT DE QUALITÉ DE LA DONNÉE (Tableau récapitulatif) ---
quality_audit = pd.DataFrame(
    {
        "Variable": df.columns,
        "Type_Donnée": df.dtypes,
        "Nb_Valeurs_Renseignées": df.count(),
        "Nb_Valeurs_Manquantes": df.isnull().sum(),
        "%_Manquants": (df.isnull().sum() / len(df)) * 100,
        "Nb_Valeurs_Unique": df.nunique(),
    }
).sort_values(by="%_Manquants", ascending=False)


In [ ]:
print("\n--- TABLEAU D'AUDIT DE QUALITÉ (Taux de manquants & Types) ---")
afficher_tableau(quality_audit,buttons=["colvis"])

afficher_tableau(df.head())


On remarque que les variables suivantes qui nous serons utiles pour la segmentation n'ont pas de valeurs manquantes et sont au bon format pour une segmentation adéquate: 
- "nb_tiers" qui represente le nombre de tiers de l'obligor et 
- "cod_ax_unite_min" qui représente le numéro du compte de plus bas niveau détenu par l'obligor (1 pour compte de client standard, 2 pour compte client de banque privée et  3 pour compte client de banque de gestion de fortune)

## Segmentation des données

## Segmentation du portefeuille

### Objectif

Un modèle de PD n'est fiable que s'il est construit sur des populations **homogènes en termes de risque** et **de taille suffisante** pour en estimer les paramètres de façon robuste. La segmentation retenue doit donc concilier deux exigences, conformément à l'esprit des orientations de l'EBA sur l'estimation des PD (EBA/GL/2017/16) :

- **la pertinence métier** : regrouper des emprunteurs dont le profil de risque, la relation bancaire et les mécanismes de défaut sont comparables ;
- **la robustesse statistique** : garantir dans chaque segment un nombre d'observations, et surtout de défauts, suffisant pour estimer et valider un modèle.

### Premier axe : particuliers standards vs BPGF

Le premier axe distingue la clientèle **standard** de la clientèle **Banque Privée / Gestion de Fortune (BPGF)**. Il s'appuie sur la variable `cod_axe_unite_max`, qui donne le niveau de compte le plus élevé détenu par l'obligor :

| Code | Type de compte |
|:---:|---|
| 1 | Compte client standard |
| 2 | Compte client Banque Privée |
| 3 | Compte client Gestion de Fortune |

**Règle retenue : un obligor est classé BPGF dès qu'il détient au moins un compte en Banque Privée ou en Gestion de Fortune (`cod_axe_unite_max >= 2`).**

Cette règle se justifie sur le plan métier :

- **C'est le compte le plus élevé qui définit la relation client.** Un client fortuné qui détient aussi un compte standard (pour ses opérations courantes, par exemple) reste un client patrimonial. Son profil de risque dépend de son patrimoine global, de la diversité de ses actifs et de l'accompagnement dédié dont il bénéficie, pas de la nature de son compte le plus simple.
- **Les profils de risque sont très différents.** La clientèle BPGF a en général des revenus plus élevés et plus diversifiés, un patrimoine mobilisable important et des crédits souvent adossés à des actifs (nantissements, financements patrimoniaux). Son comportement de défaut n'est donc pas comparable à celui des particuliers standards.
- **Les processus de gestion sont distincts.** Ces clients relèvent de filières commerciales, de circuits d'octroi et de suivi du risque spécifiques, ce qui justifie de les traiter dans un segment à part.

### Second axe : mono-tiers vs multi-tiers

Au sein de chaque population, on distingue ensuite les dossiers **mono-tiers** (un seul emprunteur) des dossiers **multi-tiers** (co-emprunteurs, comme un couple ou plusieurs associés).

Cette distinction est pertinente pour le risque : un dossier multi-tiers mutualise les revenus et la responsabilité du remboursement entre plusieurs personnes, ce qui modifie la capacité de remboursement et la probabilité de défaut par rapport à un emprunteur seul.

### Contrainte d'effectifs et segmentation finale

Croiser les deux axes donnerait quatre segments. Mais la population **BPGF ne compte que 31 observations**. La subdiviser en mono-tiers et multi-tiers produirait des sous-segments avec très peu de défauts pour estimer un taux de défaut fiable ou construire un modèle.

Le second axe n'est donc appliqué qu'à la population standard, qui est assez volumineuse en terme de nombre de défauts. **La segmentation finale compte trois classes :**

| Segment | Définition |
|---|---|
| **BPGF** | Au moins un compte en Banque Privée ou Gestion de Fortune (`cod_axe_unite_max >= 2`) |
| **Non BPGF – Mono-tiers** | Clientèle standard, un seul emprunteur |
| **Non BPGF – Multi-tiers** | Clientèle standard, plusieurs co-emprunteurs |

Ce découpage conserve la distinction métier la plus structurante (standard vs patrimonial), exploite la distinction mono/multi-tiers là où les effectifs le permettent, et évite de créer des segments trop petits pour donner des estimations fiables.

### Limite

Même regroupé en un seul segment, le nombre de défaut dans BPGF reste de très faible. Pour ce segment, une approche statistique classique sera probablement **peu robuste**. Il faudra envisager une **approche adaptée aux portefeuilles à faible taux de défaut** (*low default portfolio*) comme une augmentation de données.

In [ ]:
df=df.copy()
# 1. Création de la variable BPGF (1 si cod_axe_unite >= 2, 0 sinon)
df["BPGF"] = (df["cod_axe_unite_max"] >= 2).astype(int)

# 2. Création de la variable segment_tiers ('multi' si nb_tiers >= 2, 'mono' sinon)
df["segment_tiers"] = np.where(df["nb_tiers"] >= 2, "multi", "mono")


# 3. Création des 4 segments d'individus
# Définition des conditions
conditions = [
    (df["BPGF"] == 0) & (df["segment_tiers"] == "mono"),
    (df["BPGF"] == 0) & (df["segment_tiers"] == "multi"),
    (df["BPGF"] == 1) & (df["segment_tiers"] == "mono"),
    (df["BPGF"] == 1) & (df["segment_tiers"] == "multi"),
]

# Libellés des 4 classes
labels = [
    "1. Non-BPGF / Mono-tiers",
    "2. Non-BPGF / Multi-tiers",
    "3. BPGF / Mono-tiers",
    "4. BPGF / Multi-tiers",
]

df["segment_4_classes"] = np.select(conditions, labels, default="Inconnu")

In [ ]:
#répartition de la variable de défaut
afficher_tableau(df["ddefaut_ndb"].value_counts(dropna= False), buttons=["colvis"])

# Vérification du taux de défaut dans chaque groupe

In [ ]:
tableau_cible(df, var="segment_4_classes", titre="Répartition du défaut par segment")
graphique_taux_defaut(df, "segment_4_classes")

On remarque que la division en mono et multi classe dans le groupe BPGF donne des classes avec un très petit nombre de défaut. Donc la segmentation multi-tiers/mono-tier ne sera pas faite dans la classe BPGF. On a finalement 3 classes: BPGF, Non-BPGF et mono_tiers, Non_BPGF et multi-tiers.

In [ ]:
#  Définition des 3 bases issues de la segmentation
df_bpgf = df[df["BPGF"] == 1]
df_non_bpgf_mono = df[(df["BPGF"] == 0) & (df["segment_tiers"] == "mono")]
df_non_bpgf_multi = df[(df["BPGF"] == 0) & (df["segment_tiers"] == "multi")]

# Exportation vers un unique fichier Excel avec 3 feuilles
output_processed = PROJECT_DIR/"data"/"processed"

df_bpgf.to_csv(
    output_processed/"df_bpgf.csv", index=False
)
df_non_bpgf_mono.to_csv(
    output_processed/"df_non_bpgf_mono.csv", index=False
)
df_non_bpgf_multi.to_csv(
   output_processed/"df_non_bpgf_multi.csv", index=False
)